In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('dataset_sensor_iot_kelas - dataset_sensor_iot_kelas.csv')
df.head()

,id_bacaan,waktu,ruang,suhu_celsius,kelembapan_persen,status_sensor
0,SNS0018,4 Agustus 2026 jam 13:00,LAB-KOMPUTER-1,29.1,NaN,Nonaktif
1,SNS0051,2026-08-08 7:00,kelas xi rpl 3,24.8,64.0,Aktif
2,SNS0030,05/08/2026 15.00,XI-RPL-3,30.3,45.0,AKTIF
3,SNS0015,2026-08-04 11:00,Kelas Lab Komputer 2,26.6 derajat,62.0,nonaktif
4,SNS0038,2026-08-06 13:00,Kelas XI RPL 1,27.0C,59.0,nonaktif


Dapat terlihat bahwa data tiap baris tidak rapi dan belum di standarisasikan

In [5]:
df.describe()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 74 entries, 0 to 73
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id_bacaan          74 non-null     object 
 1   waktu              74 non-null     object 
 2   ruang              74 non-null     object 
 3   suhu_celsius       70 non-null     object 
 4   kelembapan_persen  71 non-null     float64
 5   status_sensor      71 non-null     object 
dtypes: float64(1), object(5)
memory usage: 3.6+ KB


In [6]:
print(df.isnull().sum())

id_bacaan            0
waktu                0
ruang                0
suhu_celsius         4
kelembapan_persen    3
status_sensor        3
dtype: int64


Dapat terlihat bahwa terdapat beberapa data null di kolom kelembapan_persen dan kolom status_sensor

In [ ]:
import re

df_clean = df.copy()

# Membersihkan spasi pada kolom teks
for kolom in ['id_bacaan', 'waktu', 'ruang', 'status_sensor']:
    df_clean[kolom] = df_clean[kolom].astype('string').str.strip()

# Menyamakan nama ruang
def normalisasi_ruang(nama):
    nama = nama.upper().replace(' ', '')
    if 'LAB-KOMPUTER-1' in nama or 'KELASLABKOMPUTER1' in nama:
        return 'LAB-KOMPUTER-1'
    if 'LAB-KOMPUTER-2' in nama or 'KELASLABKOMPUTER2' in nama:
        return 'LAB-KOMPUTER-2'
    if 'XIRPL1' in nama or 'KELASXIRPL1' in nama:
        return 'XI-RPL-1'
    if 'XIRPL2' in nama or 'KELASXIRPL2' in nama:
        return 'XI-RPL-2'
    if 'XIRPL3' in nama or 'KELASXIRPL3' in nama:
        return 'XI-RPL-3'
    return nama

df_clean['ruang'] = df_clean['ruang'].map(normalisasi_ruang)

# Mengubah suhu dan kelembapan menjadi angka
df_clean['suhu_celsius'] = pd.to_numeric(
    df_clean['suhu_celsius'].astype('string').str.extract(r'([0-9]+(?:\.[0-9]+)?)')[0],
    errors='coerce'
)
df_clean['kelembapan_persen'] = pd.to_numeric(df_clean['kelembapan_persen'], errors='coerce')

# Nilai kelembapan harus berada di antara 0 dan 100
df_clean.loc[~df_clean['kelembapan_persen'].between(0, 100), 'kelembapan_persen'] = np.nan

# Menyamakan status sensor
df_clean['status_sensor'] = (
    df_clean['status_sensor'].str.lower().map({
        'aktif': 'Aktif',
        'nonaktif': 'Nonaktif'
    })
)

In [ ]:
bulan = {
    'januari': '01', 'februari': '02', 'maret': '03', 'april': '04',
    'mei': '05', 'juni': '06', 'juli': '07', 'agustus': '08',
    'september': '09', 'oktober': '10', 'november': '11', 'desember': '12'
}

def normalisasi_waktu(waktu):
    waktu = waktu.lower().replace('.', ':')
    for nama_bulan, nomor_bulan in bulan.items():
        waktu = waktu.replace(nama_bulan, nomor_bulan)
    waktu = waktu.replace('jam', '')
    waktu = re.sub(r'\s+', ' ', waktu).strip()
    return pd.to_datetime(waktu, dayfirst=True, errors='coerce')

df_clean['waktu'] = df_clean['waktu'].map(normalisasi_waktu)

# Mengisi data kosong dengan nilai tengah atau nilai yang paling sering muncul
df_clean['suhu_celsius'] = df_clean['suhu_celsius'].fillna(df_clean['suhu_celsius'].median())
df_clean['kelembapan_persen'] = df_clean['kelembapan_persen'].fillna(
    df_clean['kelembapan_persen'].median()
)
df_clean['status_sensor'] = df_clean['status_sensor'].fillna('Tidak diketahui')

# Menghapus baris yang sama persis
df_clean = df_clean.drop_duplicates().reset_index(drop=True)

df_clean.info()
df_clean.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 70 entries, 0 to 69
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   id_bacaan          70 non-null     string        
 1   waktu              70 non-null     datetime64[ns]
 2   ruang              70 non-null     object        
 3   suhu_celsius       70 non-null     Float64       
 4   kelembapan_persen  70 non-null     float64       
 5   status_sensor      70 non-null     object        
dtypes: Float64(1), datetime64[ns](1), float64(1), object(2), string(1)
memory usage: 3.5+ KB


,id_bacaan,waktu,ruang,suhu_celsius,kelembapan_persen,status_sensor
0,SNS0018,2026-08-04 13:00:00,LAB-KOMPUTER-1,29.1,61.5,Nonaktif
1,SNS0051,2026-08-08 07:00:00,XI-RPL-3,24.8,64.0,Aktif
2,SNS0030,2026-08-05 15:00:00,XI-RPL-3,30.3,45.0,Aktif
3,SNS0015,2026-08-04 11:00:00,LAB-KOMPUTER-2,26.6,62.0,Nonaktif
4,SNS0038,2026-08-06 13:00:00,XI-RPL-1,27.0,59.0,Nonaktif


In [11]:
print('Jumlah nilai null:')
print(df_clean.isnull().sum())
print('\nJumlah baris duplikat:', df_clean.duplicated().sum())
print('\nRentang kelembapan:', df_clean['kelembapan_persen'].min(), '-', df_clean['kelembapan_persen'].max())
print('\nKategori status sensor:')
print(df_clean['status_sensor'].value_counts())

Jumlah nilai null:
id_bacaan            0
waktu                0
ruang                0
suhu_celsius         0
kelembapan_persen    0
status_sensor        0
dtype: int64

Jumlah baris duplikat: 0

Rentang kelembapan: 45.0 - 75.0

Kategori status sensor:
status_sensor
Aktif              36
Nonaktif           31
Tidak diketahui     3
Name: count, dtype: int64
